# CSV vs. Parquet con PySpark

Genera un millón de pedidos ficticios, guarda los mismos datos en CSV y Parquet y compara cuánto ocupan, qué tipos recupera Spark y qué resultado da la misma consulta. Antes de empezar, completa la [guía de entorno de PySpark](../01_environment_pyspark.md).

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("CSV vs Parquet").getOrCreate()
carpeta = (Path.cwd() / "../data/pyspark_pedidos").resolve()
carpeta.mkdir(exist_ok=True)
print(f"Archivos de la práctica: {carpeta}")

## 1. Generar los pedidos

Cada fila tiene un identificador, una zona y una cantidad de unidades.

Ese código crea un DataFrame con un millón de pedidos ficticios. Cada fila representa un pedido y tiene tres columnas: pedido, zona y unidades.
1. ``spark.range(1_000_000, numPartitions=4)`` genera una columna id con números del 0 al 999 999. Spark reparte las filas en cuatro particiones para procesarlas por bloques.
2. ``F.col("id").alias("pedido")`` usa el id como identificador del pedido y le da el nombre pedido.
3. ``F.concat(F.lit("zona_"), F.col("id") % 20).alias("zona")`` calcula el resto de dividir el id entre 20 y lo une al texto zona_. Así, las zonas se repiten de zona_0 a zona_19. ``F.lit("zona_")`` indica que ese texto es fijo.
4. ``(F.col("id") % 12 + 1).alias("unidades")`` calcula el resto de dividir el id entre 12 y le suma 1. Así, cada pedido tiene entre 1 y 12 unidades.

In [ ]:
pedidos = spark.range(1_000_000, numPartitions=4).select(
    F.col("id").alias("pedido"),
    F.concat(F.lit("zona_"), F.col("id") % 20).alias("zona"),
    (F.col("id") % 12 + 1).alias("unidades"),
)
pedidos.show(5)

## 2. Guardar los pedidos y comparar tamaños

Guarda los mismos pedidos en CSV y Parquet. Con `coalesce(1)`, Spark escribe un solo archivo de datos en cada carpeta.

In [ ]:
ruta_csv = carpeta / "pedidos_csv"
ruta_parquet = carpeta / "pedidos_parquet"

pedidos.coalesce(1).write.csv(str(ruta_csv), header=True, mode="overwrite")
pedidos.coalesce(1).write.parquet(str(ruta_parquet), mode="overwrite")

Busca el archivo por su extensión dentro de cada carpeta. `stat().st_size` indica cuántos bytes ocupa. ¿Cuál de los dos ocupa menos?

In [ ]:
archivo_csv = list(ruta_csv.glob("*.csv"))[0]
archivo_parquet = list(ruta_parquet.glob("*.parquet"))[0]

print("CSV:", archivo_csv.stat().st_size, "bytes")
print("Parquet:", archivo_parquet.stat().st_size, "bytes")

## 3. Volver a leer y comparar esquemas

CSV guarda texto. Usamos `inferSchema=True` para que Spark deduzca los tipos a partir de los valores. Parquet almacena los tipos junto con los datos. Compara cómo aparece `unidades` en cada esquema.

In [ ]:
pedidos_csv = spark.read.csv(str(ruta_csv), header=True, inferSchema=True)
pedidos_parquet = spark.read.parquet(str(ruta_parquet))

print("CSV")
pedidos_csv.printSchema()
print("Parquet")
pedidos_parquet.printSchema()

## 4. Sumar las unidades de `zona_3`

Aplica el mismo filtro y la misma suma a los dos DataFrames. Ambos deben dar **400 000 unidades**.

In [ ]:
total_csv = pedidos_csv.filter(F.col("zona") == "zona_3").agg(F.sum("unidades")).first()[0]
total_parquet = pedidos_parquet.filter(F.col("zona") == "zona_3").agg(F.sum("unidades")).first()[0]

print(f"CSV:     {total_csv} unidades")
print(f"Parquet: {total_parquet} unidades")
assert total_csv == total_parquet == 400_000

In [ ]:
spark.stop()

## Navegación

[Anterior · Ejercicios de DataFrames](04_df_exercise.ipynb) · [Volver al README](../README.md) · [Siguiente · Parquet con pandas y PySpark](06_parquet_pandas_vs_pyspark.ipynb)